<a href="https://colab.research.google.com/github/malihasaeed/langraphai/blob/main/healthcare_readmission_risk_monitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Healthcare Readmission Risk Monitor — What this multi-agent system does

Ingests a patient’s discharge record and recent clinical history (demographics, diagnoses, labs, medications, prior visits).

Checks whether required fields are present and flags missing or inconsistent data before analysis.

Retrieves supporting context (hospital policy notes or readmission risk factors) to ground the reasoning when needed.

Estimates the patient’s 30-day readmission risk using a simple risk model or rule-based scoring (as a prototype).

Assigns a risk category (low / medium / high) based on defined thresholds.

Generates an explanation of the main factors contributing to the risk score in clinician-friendly language.

Produces a structured report with: risk score, risk level, key drivers, and suggested follow-up actions (decision support only).

Applies safety checks and confidence thresholds; routes high-risk or low-confidence cases to human review instead of acting autonomously.

Uses a supervisor agent to coordinate all steps via a state-driven workflow, with controlled retries and full traceability of decisions.

In short : This multi-agent system supports hospitals by identifying patients at risk of 30-day readmission using structured clinical data and explainable risk analysis.
It coordinates specialized agents through a state-driven workflow to ensure safe, traceable, and human-reviewed decision support rather than autonomous medical action.


In [1]:
!pip install -U langgraph langchain langchain-openai pydantic duckduckgo-search wikipedia tenacity -q


langgraph: builds the multi-step agent workflow as a state machine (nodes + routing + traceability).

langchain / langchain-openai: easy way to call OpenAI models (ChatOpenAI) and handle prompts/messages.

pydantic: enforces structure (so your “patient data”, “risk output”, and “report” follow a strict format). Reduces messy outputs.

tenacity: safe, controlled retries if an API call fails (no infinite looping).

duckduckgo-search: lets the Research agent do lightweight web search (optional, but useful).

wikipedia: easy fallback tool for basic medical concept lookups (optional).

typing imports: helps define a clear shared state type (better readability + fewer bugs).

MemorySaver: stores the graph state during a run (helps tracing/debugging).

In [2]:
import os, time, json
import operator
from typing import TypedDict, List, Dict, Any, Optional, Literal, Annotated

from google.colab import userdata
from pydantic import BaseModel, Field, ValidationError

from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver


In [3]:
# Step 1: Load OpenAI API key safely from Colab Secrets
OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError(
        "OPENAI_API_KEY not found in Colab Secrets. "
        "Add it via Colab → Secrets (🔑) as OPENAI_API_KEY."
    )

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

# Step 2: Initialize the LLM (keep deterministic for consistent outputs)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("Setup complete. LLM initialized.")


Setup complete. LLM initialized.


What this step gives you

A standard patient record format (PatientRecord)

A standard risk output format (RiskAssessment)

A standard clinician report format (ClinicianReport)

One shared ReadmissionState that agents will update

In [4]:
# =========================
# Step 2: Define State + Schemas
# =========================

# 1) Patient input schema (structured + validated)
class PatientRecord(BaseModel):
    patient_id: str = Field(..., description="Unique patient identifier")
    age: int = Field(..., ge=0, le=120, description="Patient age")
    sex: Literal["female", "male", "other", "unknown"] = Field("unknown")

    # recent utilization
    prior_admissions_6mo: int = Field(0, ge=0, description="Admissions in last 6 months")
    er_visits_6mo: int = Field(0, ge=0, description="ER visits in last 6 months")
    length_of_stay_days: int = Field(0, ge=0, description="Length of stay for the current admission")

    # clinical complexity (simple prototype fields)
    chronic_conditions_count: int = Field(0, ge=0, description="Count of chronic conditions")
    medications_count: int = Field(0, ge=0, description="Active medications at discharge")

    # discharge context
    discharge_disposition: Literal[
        "home", "home_with_services", "skilled_nursing", "rehab", "other", "unknown"
    ] = Field("unknown")

    # optional notes (kept short)
    notes: Optional[str] = Field(None, description="Short discharge notes or relevant comments")


# 2) Model output schema (risk score + confidence + reasons)
class RiskAssessment(BaseModel):
    risk_score: float = Field(..., ge=0.0, le=1.0, description="Probability-like risk score (0 to 1)")
    risk_bucket: Literal["low", "medium", "high"] = Field(..., description="Risk category")
    confidence: Literal["low", "medium", "high"] = Field(..., description="Model confidence level")
    top_risk_factors: List[str] = Field(default_factory=list, description="Top factors driving risk")


# 3) Final report schema (what a clinician sees)
class ClinicianReport(BaseModel):
    summary: str = Field(..., description="2-4 sentence clinician-friendly summary")
    recommended_actions: List[str] = Field(default_factory=list, description="Suggested follow-up actions (non-prescriptive)")
    requires_human_review: bool = Field(..., description="True if high risk, low confidence, or missing critical data")
    disclaimers: List[str] = Field(default_factory=list, description="Safety disclaimers / scope limits")


# 4) Shared LangGraph state (all agents read/write this)
class ReadmissionState(TypedDict, total=False):
    # inputs
    patient: Dict[str, Any]                 # raw dict input
    patient_validated: PatientRecord        # validated Pydantic object

    # retrieval (optional)
    retrieved_context: str
    sources: List[str]

    # outputs
    risk: Dict[str, Any]                    # raw dict output
    risk_validated: RiskAssessment          # validated Pydantic object
    report: Dict[str, Any]                  # raw dict output
    report_validated: ClinicianReport       # validated Pydantic object

    # control + safety
    questions_for_user: List[str]
    validation_errors: List[str]
    requires_human_review: bool
    retry_count: int

print("State + schemas defined successfully.")


State + schemas defined successfully.


Clarify Agent first (super useful), which:

checks missing critical info

writes questions_for_user

sets requires_human_review if needed

In [5]:
# =========================
# Step 3: Clarify Agent (missing info handler)
# =========================
# Why we build this first:
# - It prevents the system from guessing when data is missing.
# - It adds safety: if critical fields are missing, we ask questions or route to human review.
# - It keeps the workflow reliable (LangGraph mindset).

CRITICAL_FIELDS = [
    "patient_id",
    "age",
    "prior_admissions_6mo",
    "er_visits_6mo",
    "length_of_stay_days",
    "chronic_conditions_count",
    "medications_count",
    "discharge_disposition",
]

def _is_missing(value: Any) -> bool:
    """Simple helper: decide if a field is missing/invalid."""
    if value is None:
        return True
    if isinstance(value, str) and value.strip() == "":
        return True
    return False

def clarify_agent(state: ReadmissionState) -> ReadmissionState:
    """
    Clarify Agent:
    - Checks if patient dict exists
    - Checks for missing critical fields
    - Writes questions_for_user
    - Sets requires_human_review when critical info is missing
    """
    patient = state.get("patient", {})

    # Ensure we always have lists in the state
    state.setdefault("questions_for_user", [])
    state.setdefault("validation_errors", [])
    state.setdefault("retry_count", 0)

    # If patient object is missing completely, ask for it
    if not patient:
        state["questions_for_user"] = [
            "Please provide patient data (patient_id, age, recent admissions/ER visits, discharge disposition, etc.)."
        ]
        state["requires_human_review"] = True
        return state

    missing_fields = []
    for f in CRITICAL_FIELDS:
        if f not in patient or _is_missing(patient.get(f)):
            missing_fields.append(f)

    if missing_fields:
        # Ask minimal, targeted questions
        questions = []
        for f in missing_fields:
            questions.append(f"Missing '{f}'. Please provide a value for '{f}'.")

        state["questions_for_user"] = questions
        state["requires_human_review"] = True  # do not proceed automatically
    else:
        # No missing fields → safe to proceed to validation/model steps
        state["questions_for_user"] = []
        state["requires_human_review"] = False

    return state

print("Clarify agent defined.")


Clarify agent defined.


testing

In [6]:
# =========================
# Quick test
# =========================

# Example: intentionally incomplete patient record
test_state: ReadmissionState = {
    "patient": {
        "patient_id": "P-001",
        "age": 67,
        # missing several fields on purpose
        "discharge_disposition": "home"
    }
}

out = clarify_agent(test_state)
print("Questions:", out.get("questions_for_user"))
print("Requires human review:", out.get("requires_human_review"))


Questions: ["Missing 'prior_admissions_6mo'. Please provide a value for 'prior_admissions_6mo'.", "Missing 'er_visits_6mo'. Please provide a value for 'er_visits_6mo'.", "Missing 'length_of_stay_days'. Please provide a value for 'length_of_stay_days'.", "Missing 'chronic_conditions_count'. Please provide a value for 'chronic_conditions_count'.", "Missing 'medications_count'. Please provide a value for 'medications_count'."]
Requires human review: True


In [7]:
# =========================
# Step 4: Validation Agent (Pydantic schema enforcement)
# =========================
# Purpose:
# - Converts raw patient dict -> PatientRecord (validated)
# - Captures validation errors instead of crashing the workflow
# - Sets requires_human_review when validation fails (safety)

def validation_agent(state: ReadmissionState) -> ReadmissionState:
    """
    Validation Agent:
    - Validates the raw patient dict using PatientRecord (Pydantic).
    - If validation succeeds: stores PatientRecord in state["patient_validated"]
    - If validation fails: stores readable errors in state["validation_errors"]
      and routes to human review by setting requires_human_review=True
    """
    state.setdefault("validation_errors", [])
    state.setdefault("retry_count", 0)

    patient = state.get("patient", {})
    if not patient:
        state["validation_errors"].append("No patient data found for validation.")
        state["requires_human_review"] = True
        return state

    try:
        validated = PatientRecord(**patient)
        state["patient_validated"] = validated
        # If we made it here, the structure is good.
        # We keep requires_human_review as-is (it may already be True from clarify step).
        state.setdefault("requires_human_review", False)
    except ValidationError as e:
        # Convert Pydantic error to simple, readable messages
        errors = []
        for err in e.errors():
            field = ".".join([str(x) for x in err.get("loc", [])])
            msg = err.get("msg", "Invalid value")
            errors.append(f"{field}: {msg}")
        state["validation_errors"].extend(errors)
        state["requires_human_review"] = True

    return state

print("Validation agent defined.")


Validation agent defined.


test it

In [8]:
# =========================
# Quick test
# =========================

# Valid example
valid_state: ReadmissionState = {
    "patient": {
        "patient_id": "P-002",
        "age": 72,
        "sex": "female",
        "prior_admissions_6mo": 1,
        "er_visits_6mo": 2,
        "length_of_stay_days": 5,
        "chronic_conditions_count": 3,
        "medications_count": 8,
        "discharge_disposition": "home_with_services",
        "notes": "Discharged with home care support."
    }
}

validated_out = validation_agent(valid_state)
print("Validated patient object:", isinstance(validated_out.get("patient_validated"), PatientRecord))
print("Errors:", validated_out.get("validation_errors"))
print("Requires human review:", validated_out.get("requires_human_review"))

# Invalid example (age too high)
invalid_state: ReadmissionState = {
    "patient": {
        "patient_id": "P-003",
        "age": 999,
        "prior_admissions_6mo": 0,
        "er_visits_6mo": 0,
        "length_of_stay_days": 1,
        "chronic_conditions_count": 0,
        "medications_count": 1,
        "discharge_disposition": "home",
    }
}

invalid_out = validation_agent(invalid_state)
print("\nInvalid case errors:", invalid_out.get("validation_errors"))
print("Requires human review:", invalid_out.get("requires_human_review"))


Validated patient object: True
Errors: []
Requires human review: False

Invalid case errors: ['age: Input should be less than or equal to 120']
Requires human review: True


In [9]:
# =========================
# Step 5: Rule-Based Risk Model Agent (prototype)
# =========================
# Why rule-based first:
# - Transparent and debuggable
# - Works without training data
#
# Output:
# - risk_score (0 to 1)
# - risk_bucket (low/medium/high)
# - confidence (low/medium/high)
# - top_risk_factors (human-readable reasons)

def _bucket_from_score(score: float) -> str:
    """Convert a risk score to a bucket using simple thresholds."""
    if score < 0.30:
        return "low"
    elif score < 0.60:
        return "medium"
    return "high"

def _confidence_from_inputs(patient: PatientRecord) -> str:
    """
    Confidence heuristic for prototype:
    - If lots of utilization/complexity signals exist, confidence is higher.
    - If most signals are near zero, keep confidence medium (less evidence).
    """
    signal_strength = (
        patient.prior_admissions_6mo
        + patient.er_visits_6mo
        + patient.chronic_conditions_count
        + (1 if patient.length_of_stay_days >= 7 else 0)
        + (1 if patient.medications_count >= 10 else 0)
    )

    if signal_strength >= 6:
        return "high"
    if signal_strength >= 2:
        return "medium"
    return "low"

def risk_model_agent_rule_based(state: ReadmissionState) -> ReadmissionState:
    """
    Rule-Based Risk Model Agent:
    - Requires state["patient_validated"] (Pydantic PatientRecord)
    - Computes a simple additive score and normalizes to 0..1
    - Stores both raw dict and validated RiskAssessment in state
    """
    state.setdefault("validation_errors", [])
    state.setdefault("retry_count", 0)

    patient_obj = state.get("patient_validated")
    if not patient_obj:
        state["validation_errors"].append("Risk model skipped: patient_validated not found.")
        state["requires_human_review"] = True
        return state

    patient: PatientRecord = patient_obj

    # ----- Scoring rules (simple and explainable) -----
    # Each rule adds points; we normalize by max_points to create 0..1 score.
    points = 0
    max_points = 14  # keep in sync with rules below

    top_factors: List[str] = []

    # Age
    if patient.age >= 75:
        points += 2
        top_factors.append("Older age (75+)")
    elif patient.age >= 65:
        points += 1
        top_factors.append("Age 65–74")

    # Prior admissions (strong signal)
    if patient.prior_admissions_6mo >= 2:
        points += 3
        top_factors.append("Multiple admissions in last 6 months (2+)")
    elif patient.prior_admissions_6mo == 1:
        points += 2
        top_factors.append("Recent admission in last 6 months (1)")

    # ER visits
    if patient.er_visits_6mo >= 3:
        points += 2
        top_factors.append("Frequent ER visits in last 6 months (3+)")
    elif patient.er_visits_6mo >= 1:
        points += 1
        top_factors.append("Recent ER visit(s) in last 6 months")

    # Length of stay
    if patient.length_of_stay_days >= 7:
        points += 2
        top_factors.append("Long hospital stay (7+ days)")
    elif patient.length_of_stay_days >= 4:
        points += 1
        top_factors.append("Moderate hospital stay (4–6 days)")

    # Chronic conditions
    if patient.chronic_conditions_count >= 4:
        points += 2
        top_factors.append("Multiple chronic conditions (4+)")
    elif patient.chronic_conditions_count >= 2:
        points += 1
        top_factors.append("More than one chronic condition (2–3)")

    # Medications (proxy for complexity)
    if patient.medications_count >= 10:
        points += 2
        top_factors.append("High medication burden (10+ meds)")
    elif patient.medications_count >= 5:
        points += 1
        top_factors.append("Moderate medication burden (5–9 meds)")

    # Discharge disposition
    if patient.discharge_disposition in ["skilled_nursing", "rehab"]:
        points += 1
        top_factors.append("Discharged to facility care (SNF/Rehab)")
    elif patient.discharge_disposition == "home_with_services":
        points += 1
        top_factors.append("Discharged home with services")

    # Normalize score to 0..1
    risk_score = max(0.0, min(1.0, points / max_points))

    risk_bucket = _bucket_from_score(risk_score)
    confidence = _confidence_from_inputs(patient)

    # Keep only top 4 factors for readability
    top_factors = top_factors[:4]

    risk_dict = {
        "risk_score": risk_score,
        "risk_bucket": risk_bucket,
        "confidence": confidence,
        "top_risk_factors": top_factors,
    }

    # Validate output structure with Pydantic
    try:
        state["risk"] = risk_dict
        state["risk_validated"] = RiskAssessment(**risk_dict)
        # if model confidence is low, push to human review
        state["requires_human_review"] = state.get("requires_human_review", False) or (confidence == "low")
    except ValidationError as e:
        state["validation_errors"].append(f"RiskAssessment validation failed: {str(e)}")
        state["requires_human_review"] = True

    return state

print("Rule-based risk model agent defined.")


Rule-based risk model agent defined.


In [10]:
# =========================
# Quick test
# =========================

test_state: ReadmissionState = {
    "patient": {
        "patient_id": "P-010",
        "age": 78,
        "sex": "male",
        "prior_admissions_6mo": 2,
        "er_visits_6mo": 3,
        "length_of_stay_days": 9,
        "chronic_conditions_count": 4,
        "medications_count": 12,
        "discharge_disposition": "home_with_services",
        "notes": "Complex discharge."
    }
}

# Run clarify -> validate -> risk model (like our pipeline will do)
s = clarify_agent(test_state)
s = validation_agent(s)
s = risk_model_agent_rule_based(s)

print("Risk:", s.get("risk"))
print("Requires human review:", s.get("requires_human_review"))
print("Errors:", s.get("validation_errors"))


Risk: {'risk_score': 1.0, 'risk_bucket': 'high', 'confidence': 'high', 'top_risk_factors': ['Older age (75+)', 'Multiple admissions in last 6 months (2+)', 'Frequent ER visits in last 6 months (3+)', 'Long hospital stay (7+ days)']}
Requires human review: False
Errors: []


In [11]:
# =========================
# Step 6: Explanation & Recommendation Agent (LLM-based)
# =========================
# Purpose:
# - Turn patient + risk assessment into a clinician-friendly summary
# - Provide non-prescriptive follow-up suggestions (decision support only)
# - Enforce safety: no medical orders, no diagnosis, no definitive claims
# - Output MUST match ClinicianReport schema (validated by Pydantic)

def _safe_json_loads(text: str) -> Dict[str, Any]:
    """
    Best-effort JSON parsing:
    - Tries to find the first JSON object in the text.
    - Helps when the model wraps JSON with extra text.
    """
    text = text.strip()
    # If it's already valid JSON
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try to extract JSON object from within the text
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start : end + 1]
        return json.loads(candidate)

    raise ValueError("Could not parse JSON from model output.")

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(min=1, max=10),
    retry=retry_if_exception_type(Exception),
)
def _llm_generate_report(payload: Dict[str, Any]) -> Dict[str, Any]:
    """
    Calls the LLM to generate a ClinicianReport JSON.
    Uses Tenacity for controlled retries on transient errors.
    """
    system_msg = (
        "You are a clinical analytics assistant. You DO NOT provide medical decisions, diagnoses, "
        "or prescriptions. You only produce decision-support summaries for clinicians.\n\n"
        "Hard rules:\n"
        "1) Use only the provided patient data and risk assessment.\n"
        "2) Do NOT invent missing information.\n"
        "3) Recommendations must be non-prescriptive (e.g., follow-up call, care coordination).\n"
        "4) Be concise and professional.\n"
        "5) Output MUST be valid JSON that matches this schema:\n"
        '{ "summary": string, "recommended_actions": [string], "requires_human_review": boolean, "disclaimers": [string] }\n'
        "Return JSON only. No extra text."
    )

    human_msg = (
        "Create a clinician-friendly readmission risk note from this input:\n\n"
        f"{json.dumps(payload, indent=2)}\n\n"
        "Remember: decision-support only, no medical orders."
    )

    resp = llm.invoke([{"role": "system", "content": system_msg},
                       {"role": "user", "content": human_msg}])
    return _safe_json_loads(resp.content)

def explanation_recommendation_agent(state: ReadmissionState) -> ReadmissionState:
    """
    Explanation & Recommendation Agent:
    - Requires validated patient + validated risk
    - Produces a ClinicianReport JSON (validated by Pydantic)
    - Sets/propagates requires_human_review based on risk + confidence + missing data flags
    """
    state.setdefault("validation_errors", [])
    state.setdefault("retry_count", 0)

    patient_obj = state.get("patient_validated")
    risk_obj = state.get("risk_validated")

    if not patient_obj or not risk_obj:
        state["validation_errors"].append(
            "Explanation agent skipped: patient_validated or risk_validated missing."
        )
        state["requires_human_review"] = True
        return state

    patient: PatientRecord = patient_obj
    risk: RiskAssessment = risk_obj

    # Build payload for the LLM using ONLY verified fields
    payload = {
        "patient": patient.model_dump(),
        "risk_assessment": risk.model_dump(),
        "notes": {
            "context": "30-day readmission risk decision-support summary",
            "requirements": [
                "Use only provided data",
                "No medical decisions or prescriptions",
                "Keep recommendations non-prescriptive",
            ],
        },
    }

    # Let the LLM draft a structured report
    try:
        report_dict = _llm_generate_report(payload)

        # Add/ensure standard disclaimers (always include these)
        default_disclaimers = [
            "Decision-support only. Not medical advice.",
            "Review by a qualified clinician is required before any action.",
            "If data is incomplete or confidence is low, interpret cautiously."
        ]
        # Merge disclaimers safely
        model_disclaimers = report_dict.get("disclaimers", [])
        if not isinstance(model_disclaimers, list):
            model_disclaimers = []
        report_dict["disclaimers"] = list(dict.fromkeys(model_disclaimers + default_disclaimers))

        # Determine human review requirement (guardrail logic)
        must_review = state.get("requires_human_review", False)

        # Force review for high risk or low confidence
        if risk.risk_bucket == "high" or risk.confidence == "low":
            must_review = True

        report_dict["requires_human_review"] = bool(must_review)

        # Validate against Pydantic schema (prevents malformed output)
        validated_report = ClinicianReport(**report_dict)

        state["report"] = report_dict
        state["report_validated"] = validated_report
        state["requires_human_review"] = validated_report.requires_human_review

    except Exception as e:
        state["validation_errors"].append(f"Explanation agent error: {str(e)}")
        state["requires_human_review"] = True

    return state

print("Explanation & Recommendation agent defined.")


Explanation & Recommendation agent defined.


In [12]:
# =========================
# Quick test
# =========================

demo_state: ReadmissionState = {
    "patient": {
        "patient_id": "P-020",
        "age": 70,
        "sex": "female",
        "prior_admissions_6mo": 1,
        "er_visits_6mo": 2,
        "length_of_stay_days": 6,
        "chronic_conditions_count": 3,
        "medications_count": 9,
        "discharge_disposition": "home_with_services",
        "notes": "Patient has limited mobility and needs support."
    }
}

demo_state = clarify_agent(demo_state)
demo_state = validation_agent(demo_state)
demo_state = risk_model_agent_rule_based(demo_state)
demo_state = explanation_recommendation_agent(demo_state)

print("Report (validated):")
print(demo_state.get("report_validated"))
print("\nRequires human review:", demo_state.get("requires_human_review"))
print("Errors:", demo_state.get("validation_errors"))


Report (validated):
summary='The patient is a 70-year-old female with a medium risk of readmission within 30 days. She has a history of 1 prior admission and 2 ER visits in the last 6 months, along with 3 chronic conditions and 9 medications. The patient has limited mobility and requires support at home.' recommended_actions=['Consider follow-up call to assess ongoing support needs.', 'Coordinate care services to assist with mobility and daily activities.', 'Monitor for any signs of deterioration or need for additional services.'] requires_human_review=False disclaimers=['This summary is for decision-support purposes only and does not constitute medical advice.', 'Decision-support only. Not medical advice.', 'Review by a qualified clinician is required before any action.', 'If data is incomplete or confidence is low, interpret cautiously.']

Requires human review: False
Errors: []


In [13]:
# =========================
# Step 7A: Update the Explanation agent to track retry_count (controlled retries)
# =========================
# Why:
# - If the LLM returns invalid JSON or a transient API error happens,
#   we retry a limited number of times (no infinite loops).

def explanation_recommendation_agent(state: ReadmissionState) -> ReadmissionState:
    """
    Explanation & Recommendation Agent (LLM-based):
    - Creates a structured ClinicianReport from validated patient + risk
    - Uses strict safety rules and returns JSON only
    - If it fails, increments retry_count (so Supervisor can decide to retry or stop)
    """
    state.setdefault("validation_errors", [])
    state.setdefault("retry_count", 0)

    patient_obj = state.get("patient_validated")
    risk_obj = state.get("risk_validated")

    if not patient_obj or not risk_obj:
        state["validation_errors"].append(
            "Explanation agent skipped: patient_validated or risk_validated missing."
        )
        state["requires_human_review"] = True
        return state

    patient: PatientRecord = patient_obj
    risk: RiskAssessment = risk_obj

    payload = {
        "patient": patient.model_dump(),
        "risk_assessment": risk.model_dump(),
        "notes": {
            "context": "30-day readmission risk decision-support summary",
            "requirements": [
                "Use only provided data",
                "No medical decisions or prescriptions",
                "Keep recommendations non-prescriptive",
            ],
        },
    }

    try:
        report_dict = _llm_generate_report(payload)

        # Always include safety disclaimers (even if model forgets)
        default_disclaimers = [
            "Decision-support only. Not medical advice.",
            "Review by a qualified clinician is required before any action.",
            "If data is incomplete or confidence is low, interpret cautiously."
        ]

        model_disclaimers = report_dict.get("disclaimers", [])
        if not isinstance(model_disclaimers, list):
            model_disclaimers = []
        # Merge without duplicates
        report_dict["disclaimers"] = list(dict.fromkeys(model_disclaimers + default_disclaimers))

        # Guardrail: require review for high risk or low confidence
        must_review = state.get("requires_human_review", False)
        if risk.risk_bucket == "high" or risk.confidence == "low":
            must_review = True

        report_dict["requires_human_review"] = bool(must_review)

        # Validate structure
        validated_report = ClinicianReport(**report_dict)

        state["report"] = report_dict
        state["report_validated"] = validated_report
        state["requires_human_review"] = validated_report.requires_human_review

    except Exception as e:
        # Controlled retry: count failures (Supervisor will decide next step)
        state["retry_count"] = state.get("retry_count", 0) + 1
        state["validation_errors"].append(f"Explanation agent error: {str(e)}")
        state["requires_human_review"] = True

    return state

print("Explanation agent updated with controlled retry_count.")


Explanation agent updated with controlled retry_count.


In [14]:
# =========================
# Step 7B: Report Agent (final formatting for display)
# =========================
# Why:
# - The explanation agent produces structured JSON (ClinicianReport).
# - This Report agent converts it into a clean, readable final output
#   (e.g., markdown text for UI / email / note).

def report_agent(state: ReadmissionState) -> ReadmissionState:
    """
    Report Agent:
    - Builds a final markdown string using validated outputs
    - Keeps it readable and consistent
    """
    state.setdefault("validation_errors", [])

    patient_obj = state.get("patient_validated")
    risk_obj = state.get("risk_validated")
    report_obj = state.get("report_validated")

    if not patient_obj or not risk_obj or not report_obj:
        state["validation_errors"].append(
            "Report agent skipped: patient_validated, risk_validated, or report_validated missing."
        )
        state["requires_human_review"] = True
        return state

    patient: PatientRecord = patient_obj
    risk: RiskAssessment = risk_obj
    report: ClinicianReport = report_obj

    final_md = f"""
## Readmission Risk Summary (Decision Support)

**Patient ID:** {patient.patient_id}
**Risk Score (0–1):** {risk.risk_score:.2f}
**Risk Level:** {risk.risk_bucket.upper()}
**Confidence:** {risk.confidence.upper()}
**Human Review Required:** {"YES" if report.requires_human_review else "NO"}

### Key Risk Factors
{chr(10).join([f"- {x}" for x in risk.top_risk_factors]) if risk.top_risk_factors else "- None identified from available data"}

### Clinician-Friendly Summary
{report.summary}

### Suggested Follow-Up Actions (Non-Prescriptive)
{chr(10).join([f"- {x}" for x in report.recommended_actions]) if report.recommended_actions else "- No actions suggested"}

### Safety Notes
{chr(10).join([f"- {x}" for x in report.disclaimers])}
""".strip()

    state["final_report_markdown"] = final_md
    return state

print("Report agent defined.")


Report agent defined.


In [15]:
# =========================
# Step 7C: Supervisor routing logic + LangGraph compilation
# =========================
# Simple routing rules:
# 1) If Clarify has questions → stop (we need user input)
# 2) If Validation errors exist after validation → stop (human review)
# 3) If Explanation failed and retry_count < 2 → retry Explanation
# 4) Otherwise → go to Report → END

MAX_RETRIES = 2

def route_after_clarify(state: ReadmissionState) -> str:
    # If we have questions, we stop here (waiting for user answers)
    if state.get("questions_for_user"):
        return "end"
    return "validate"

def route_after_validate(state: ReadmissionState) -> str:
    # If validation errors exist, stop (human review required)
    if state.get("validation_errors"):
        return "end"
    return "risk_model"

def route_after_explain(state: ReadmissionState) -> str:
    # If explanation failed and we haven't exceeded retries → try again
    errors = state.get("validation_errors", [])
    has_explain_error = any("Explanation agent error" in e for e in errors)

    if has_explain_error and state.get("retry_count", 0) < MAX_RETRIES:
        return "explain"

    # If explanation succeeded, go to report
    if state.get("report_validated"):
        return "report"

    # Otherwise, stop safely
    return "end"


# Build the graph
builder = StateGraph(ReadmissionState)

# Nodes (each node is one agent/function)
builder.add_node("clarify", clarify_agent)
builder.add_node("validate", validation_agent)
builder.add_node("risk_model", risk_model_agent_rule_based)
builder.add_node("explain", explanation_recommendation_agent)
builder.add_node("report", report_agent)

# Entry point
builder.set_entry_point("clarify")

# Edges + conditional routing (Supervisor-style control)
builder.add_conditional_edges("clarify", route_after_clarify, {
    "validate": "validate",
    "end": END
})

builder.add_conditional_edges("validate", route_after_validate, {
    "risk_model": "risk_model",
    "end": END
})

# Linear edge to explanation after risk model
builder.add_edge("risk_model", "explain")

# Conditional edge after explanation (retry or report or end)
builder.add_conditional_edges("explain", route_after_explain, {
    "explain": "explain",
    "report": "report",
    "end": END
})

# Finish
builder.add_edge("report", END)

# Optional: memory checkpointing (useful for tracing/debugging)
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

print("LangGraph compiled successfully.")


LangGraph compiled successfully.


In [16]:
# =========================
# Step 8: Research Agent (DuckDuckGo + Wikipedia)
# =========================
# Goal (simple):
# - Pull general, non-patient-specific information about readmission risk factors
# - Use it as "grounding context" for the LLM summary (NOT medical advice)
# - Store results in state["retrieved_context"] and state["sources"]

from duckduckgo_search import DDGS
import wikipedia

def research_agent(state: ReadmissionState) -> ReadmissionState:
    """
    Research Agent:
    - Uses DuckDuckGo + Wikipedia to fetch general background context
    - NOT patient-specific and NOT medical advice
    - Helps reduce hallucinations by giving the LLM grounded reference text
    """
    state.setdefault("validation_errors", [])
    state.setdefault("sources", [])

    # If we already have context, don't redo work
    if state.get("retrieved_context"):
        return state

    # We use validated risk info if available to guide the query (optional)
    risk_obj = state.get("risk_validated")
    risk_bucket = risk_obj.risk_bucket if risk_obj else "unknown"

    # Simple query set (general, safe, and not giving medical directions)
    queries = [
        "30-day hospital readmission risk factors",
        "common causes of hospital readmission older adults",
        "factors associated with hospital readmissions chronic conditions polypharmacy",
    ]

    # If risk looks high, broaden the context slightly (still general)
    if risk_bucket == "high":
        queries.append("interventions to reduce hospital readmissions care coordination follow-up")

    collected_snippets = []
    sources = []

    # --- DuckDuckGo Search ---
    try:
        with DDGS() as ddgs:
            for q in queries:
                # max_results small to keep output compact
                results = ddgs.text(q, max_results=3)
                for r in results:
                    title = r.get("title", "")
                    href = r.get("href", "")
                    body = r.get("body", "")

                    if body:
                        collected_snippets.append(f"- {title}: {body}")
                    if href:
                        sources.append(href)
    except Exception as e:
        state["validation_errors"].append(f"Research agent (DuckDuckGo) error: {str(e)}")

    # --- Wikipedia (short background) ---
    # We keep this lightweight—just general definitions, not guidance.
    try:
        # These pages usually exist, but we keep try/except to be safe.
        wiki_topics = ["Hospital readmission", "Chronic condition", "Polypharmacy"]
        for topic in wiki_topics:
            try:
                summ = wikipedia.summary(topic, sentences=2, auto_suggest=True)
                collected_snippets.append(f"- Wikipedia ({topic}): {summ}")
                sources.append(f"Wikipedia: {topic}")
            except Exception:
                # Skip if page not found or ambiguous
                pass
    except Exception as e:
        state["validation_errors"].append(f"Research agent (Wikipedia) error: {str(e)}")

    # Combine and store
    if collected_snippets:
        state["retrieved_context"] = "\n".join(collected_snippets[:12])  # keep it short
        state["sources"] = list(dict.fromkeys(sources))[:12]            # unique, short list
    else:
        state["retrieved_context"] = ""
        state["sources"] = []

    return state

print("Research agent defined.")


Research agent defined.


In [17]:
# =========================
# Step 8B: Plug Research Agent into the graph (simple routing)
# =========================
# We route: risk_model -> research -> explain
# This makes the workflow more grounded before the LLM writes the report.

# 1) Add the node
builder.add_node("research", research_agent)

# 2) Replace the old edge risk_model -> explain with:
#    risk_model -> research -> explain
# NOTE: we must rebuild edges cleanly. Easiest in Colab is to re-create the builder.
# To avoid confusion, we will rebuild the graph from scratch below.

print("Research node added (we will now rebuild & compile the graph cleanly).")


Research node added (we will now rebuild & compile the graph cleanly).


In [18]:
# =========================
# Step 8C: Rebuild + Compile the full graph cleanly (with Research)
# =========================

MAX_RETRIES = 2

def route_after_clarify(state: ReadmissionState) -> str:
    # If we have questions, stop and wait for user input
    if state.get("questions_for_user"):
        return "end"
    return "validate"

def route_after_validate(state: ReadmissionState) -> str:
    # If validation errors exist, stop (human review)
    if state.get("validation_errors"):
        return "end"
    return "risk_model"


def route_after_explain(state: ReadmissionState) -> str:
    # 1) If explanation succeeded, ALWAYS go to report
    if state.get("report_validated"):
        return "report"

    # 2) If explanation failed and retries remain, retry explanation
    errors = state.get("validation_errors", [])
    has_explain_error = any("Explanation agent error" in e for e in errors)

    if has_explain_error and state.get("retry_count", 0) < MAX_RETRIES:
        return "explain"

    # 3) Otherwise stop safely
    return "end"

print("Router updated: success -> report first.")



# Rebuild graph (fresh builder)
builder2 = StateGraph(ReadmissionState)

# Nodes
builder2.add_node("clarify", clarify_agent)
builder2.add_node("validate", validation_agent)
builder2.add_node("risk_model", risk_model_agent_rule_based)
builder2.add_node("research", research_agent)
builder2.add_node("explain", explanation_recommendation_agent)
builder2.add_node("report", report_agent)

# Entry
builder2.set_entry_point("clarify")

# Routing
builder2.add_conditional_edges("clarify", route_after_clarify, {
    "validate": "validate",
    "end": END
})

builder2.add_conditional_edges("validate", route_after_validate, {
    "risk_model": "risk_model",
    "end": END
})

# Grounding step before LLM explanation
builder2.add_edge("risk_model", "research")
builder2.add_edge("research", "explain")

builder2.add_conditional_edges("explain", route_after_explain, {
    "explain": "explain",
    "report": "report",
    "end": END
})

builder2.add_edge("report", END)

# Compile with MemorySaver (requires thread_id when invoking)
memory2 = MemorySaver()
graph2 = builder2.compile(checkpointer=memory2)

print("Graph rebuilt and compiled with Research agent.")


Router updated: success -> report first.
Graph rebuilt and compiled with Research agent.


In [19]:
def explanation_recommendation_agent(state: ReadmissionState) -> ReadmissionState:
    state.setdefault("validation_errors", [])
    state.setdefault("retry_count", 0)

    patient_obj = state.get("patient_validated")
    risk_obj = state.get("risk_validated")

    if not patient_obj or not risk_obj:
        state["validation_errors"].append(
            "Explanation agent error: patient_validated or risk_validated missing."
        )
        state["requires_human_review"] = True
        return state

    patient: PatientRecord = patient_obj
    risk: RiskAssessment = risk_obj

    payload = {
        "patient": patient.model_dump(),
        "risk_assessment": risk.model_dump(),
        "retrieved_context": state.get("retrieved_context", ""),
        "sources": state.get("sources", []),
        "notes": {
            "context": "30-day readmission risk decision-support summary",
            "requirements": [
                "Use only provided patient data + retrieved_context",
                "No medical decisions or prescriptions",
                "Keep recommendations non-prescriptive",
                "Return JSON only",
            ],
        },
    }

    try:
        report_dict = _llm_generate_report(payload)

        # Always include safety disclaimers
        default_disclaimers = [
            "Decision-support only. Not medical advice.",
            "Review by a qualified clinician is required before any action.",
            "If data is incomplete or confidence is low, interpret cautiously."
        ]
        model_disclaimers = report_dict.get("disclaimers", [])
        if not isinstance(model_disclaimers, list):
            model_disclaimers = []
        report_dict["disclaimers"] = list(dict.fromkeys(model_disclaimers + default_disclaimers))

        # Guardrail for human review
        must_review = state.get("requires_human_review", False)
        if risk.risk_bucket == "high" or risk.confidence == "low":
            must_review = True
        report_dict["requires_human_review"] = bool(must_review)

        # Validate report schema
        validated_report = ClinicianReport(**report_dict)

        # ✅ SUCCESS: clear old explanation errors + reset retry_count
        state["validation_errors"] = [
            e for e in state["validation_errors"] if "Explanation agent error" not in e
        ]
        state["retry_count"] = 0

        state["report"] = report_dict
        state["report_validated"] = validated_report
        state["requires_human_review"] = validated_report.requires_human_review

    except Exception as e:
        state["retry_count"] = state.get("retry_count", 0) + 1
        state["validation_errors"].append(f"Explanation agent error: {str(e)}")
        state["requires_human_review"] = True

    return state

print("Explanation agent updated: clears old errors on success.")


Explanation agent updated: clears old errors on success.


In [20]:
# Rebuild graph fresh so updated functions are used
builder3 = StateGraph(ReadmissionState)

builder3.add_node("clarify", clarify_agent)
builder3.add_node("validate", validation_agent)
builder3.add_node("risk_model", risk_model_agent_rule_based)
builder3.add_node("research", research_agent)
builder3.add_node("explain", explanation_recommendation_agent)
builder3.add_node("report", report_agent)

builder3.set_entry_point("clarify")

builder3.add_conditional_edges("clarify", route_after_clarify, {
    "validate": "validate",
    "end": END
})

builder3.add_conditional_edges("validate", route_after_validate, {
    "risk_model": "risk_model",
    "end": END
})

builder3.add_edge("risk_model", "research")
builder3.add_edge("research", "explain")

builder3.add_conditional_edges("explain", route_after_explain, {
    "explain": "explain",
    "report": "report",
    "end": END
})

builder3.add_edge("report", END)

memory3 = MemorySaver()
graph3 = builder3.compile(checkpointer=memory3)

print("Graph recompiled successfully as graph3.")


Graph recompiled successfully as graph3.


In [21]:
# =========================
# Step 8E: Run end-to-end with Research Agent
# =========================

initial_state: ReadmissionState = {
    "patient": {
        "patient_id": "P-200",
        "age": 76,
        "sex": "female",
        "prior_admissions_6mo": 2,
        "er_visits_6mo": 1,
        "length_of_stay_days": 8,
        "chronic_conditions_count": 4,
        "medications_count": 11,
        "discharge_disposition": "home_with_services",
        "notes": "Patient lives alone and needs support."
    },
    "retry_count": 0,
    "validation_errors": [],
    "questions_for_user": []
}

config = {"configurable": {"thread_id": "readmission-demo-002"}}
result = graph2.invoke(initial_state, config=config)

# Print research context (optional)
if result.get("retrieved_context"):
    print("\n--- Retrieved Context (short) ---")
    print(result["retrieved_context"][:800], "...\n")

# Print final report
if result.get("final_report_markdown"):
    print(result["final_report_markdown"])

# Print questions/errors if any
if result.get("questions_for_user"):
    print("\nSystem needs more info:")
    for q in result["questions_for_user"]:
        print("-", q)

if result.get("validation_errors"):
    print("\nValidation/Runtime Errors:")
    for e in result["validation_errors"]:
        print("-", e)


/tmp/ipython-input-2667284813.py:46: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use time


--- Retrieved Context (short) ---
- Wikipedia (Hospital readmission): A hospital readmission is an episode when a patient who had been discharged from a hospital is admitted again within a specified time interval. Readmission rates have increasingly been used as an outcome measure in health services research and as a quality benchmark for health systems.
- Wikipedia (Chronic condition): A chronic condition (also known as chronic disease or chronic illness) is a health condition or disease that is persistent or otherwise long-lasting in its effects or a disease that comes with time. The term chronic is often applied when the course of the disease lasts for more than three months.
- Wikipedia (Polypharmacy): Polypharmacy (polypragmasia) is an umbrella term to describe the simultaneous use of multiple medicines by a patient f ...



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [24]:
# =========================
# FIX: Add final_report_markdown to the state schema
# =========================

class ReadmissionState(TypedDict, total=False):
    # inputs
    patient: Dict[str, Any]
    patient_validated: PatientRecord

    # retrieval
    retrieved_context: str
    sources: List[str]

    # outputs
    risk: Dict[str, Any]
    risk_validated: RiskAssessment
    report: Dict[str, Any]
    report_validated: ClinicianReport

    # ✅ ADD THIS (so LangGraph keeps it)
    final_report_markdown: str

    # control + safety
    questions_for_user: List[str]
    validation_errors: List[str]
    requires_human_review: bool
    retry_count: int

print("ReadmissionState updated with final_report_markdown.")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

ReadmissionState updated with final_report_markdown.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [25]:
# =========================
# Rebuild & compile graph with updated ReadmissionState
# =========================

memory_fix = MemorySaver()

builder_fix = StateGraph(ReadmissionState)

builder_fix.add_node("clarify", clarify_agent)
builder_fix.add_node("validate", validation_agent)
builder_fix.add_node("risk_model", risk_model_agent_rule_based)
builder_fix.add_node("research", research_agent)
builder_fix.add_node("explain", explanation_recommendation_agent)
builder_fix.add_node("report", report_agent)

builder_fix.set_entry_point("clarify")

builder_fix.add_conditional_edges("clarify", route_after_clarify, {
    "validate": "validate",
    "end": END
})

builder_fix.add_conditional_edges("validate", route_after_validate, {
    "risk_model": "risk_model",
    "end": END
})

builder_fix.add_edge("risk_model", "research")
builder_fix.add_edge("research", "explain")

builder_fix.add_conditional_edges("explain", route_after_explain, {
    "explain": "explain",
    "report": "report",
    "end": END
})

builder_fix.add_edge("report", END)

graph_fix = builder_fix.compile(checkpointer=memory_fix)

print("Graph compiled successfully as graph_fix.")


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Graph compiled successfully as graph_fix.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [26]:
dummy_patient_complete = {
    "patient_id": "P-401",
    "age": 78,
    "sex": "female",
    "prior_admissions_6mo": 2,
    "er_visits_6mo": 2,
    "length_of_stay_days": 8,
    "chronic_conditions_count": 4,
    "medications_count": 11,
    "discharge_disposition": "home_with_services",
    "notes": "Lives alone, needs support with mobility."
}

initial_state_complete = {
    "patient": dummy_patient_complete,
    "retry_count": 0,
    "validation_errors": [],
    "questions_for_user": []
}

config = {"configurable": {"thread_id": "readmission-demo-final-001"}}
result = graph_fix.invoke(initial_state_complete, config=config)

print("RISK OUTPUT:", result.get("risk"))
print("\nHUMAN REVIEW REQUIRED:", result.get("requires_human_review"))
print("\nFINAL REPORT:\n")
print(result.get("final_report_markdown", "No report generated."))
print("\nERRORS:", result.get("validation_errors"))


Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

RISK OUTPUT: {'risk_score': 0.9285714285714286, 'risk_bucket': 'high', 'confidence': 'high', 'top_risk_factors': ['Older age (75+)', 'Multiple admissions in last 6 months (2+)', 'Recent ER visit(s) in last 6 months', 'Long hospital stay (7+ days)']}

HUMAN REVIEW REQUIRED: True

FINAL REPORT:

## Readmission Risk Summary (Decision Support)

**Patient ID:** P-401
**Risk Score (0–1):** 0.93
**Risk Level:** HIGH
**Confidence:** HIGH
**Human Review Required:** YES

### Key Risk Factors
- Older age (75+)
- Multiple admissions in last 6 months (2+)
- Recent ER visit(s) in last 6 months
- Long hospital stay (7+ days)

### Clinician-Friendly Summary
The patient is a 78-year-old female with a high readmission risk score of 0.93. She has had 2 prior admissions and 2 ER visits in the last 6 months, along with a long hospital stay of 8 days. She has 4 chronic conditions and is on 11 medications, indicating potential polypharmacy. The patient lives alone and requires support with mobility, which ma

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

```markdown
# Healthcare Readmission Risk Monitor

This multi-agent system is designed to support hospitals by identifying patients at risk of 30-day readmission. It uses structured clinical data and explainable risk analysis to provide decision support to clinicians.

## System Capabilities:

*   **Patient Data Ingestion**: Processes a patient’s discharge record and recent clinical history (demographics, diagnoses, labs, medications, prior visits).
*   **Data Validation**: Checks for missing or inconsistent data and flags them before analysis.
*   **Context Retrieval**: Retrieves supporting context such as hospital policy notes or readmission risk factors to ground the reasoning.
*   **Risk Estimation**: Estimates a patient’s 30-day readmission risk using a rule-based scoring model.
*   **Risk Categorization**: Assigns a risk category (low / medium / high) based on defined thresholds.
*   **Explanation Generation**: Provides a clear explanation of the main factors contributing to the risk score in clinician-friendly language.
*   **Structured Reporting**: Produces a structured report containing the risk score, risk level, key drivers, and suggested follow-up actions.
*   **Safety Checks**: Applies safety checks and confidence thresholds, routing high-risk or low-confidence cases for human review rather than autonomous action.
*   **Workflow Coordination**: Uses a supervisor agent to coordinate all steps via a state-driven workflow, ensuring controlled retries and full traceability of decisions.

## How it works:

1.  **Clarify Agent**: Handles missing critical information, asks clarifying questions, and flags cases requiring human review.
2.  **Validation Agent**: Validates raw patient data against a predefined schema (`PatientRecord`), captures errors, and routes invalid cases for review.
3.  **Rule-Based Risk Model Agent**: Calculates a 30-day readmission risk score and assigns a risk bucket (low/medium/high) and confidence level based on patient data.
4.  **Research Agent**: Retrieves general, non-patient-specific background context from sources like DuckDuckGo and Wikipedia to ground the LLM's explanations.
5.  **Explanation & Recommendation Agent**: Generates a clinician-friendly summary and non-prescriptive follow-up actions, ensuring compliance with `ClinicianReport` schema and safety guardrails.
6.  **Report Agent**: Formats the final output into a readable markdown string for display.
7.  **Supervisor Agent**: Orchestrates the entire workflow, manages state transitions, handles retries, and ensures safe progression through the system.

This system is designed for decision-support and does not provide medical advice or autonomous medical actions. All high-risk or low-confidence cases are explicitly routed for human clinician review.

In [27]:
%%writefile README.md
# Healthcare Readmission Risk Monitor

This multi-agent system is designed to support hospitals by identifying patients at risk of 30-day readmission. It uses structured clinical data and explainable risk analysis to provide decision support to clinicians.

## System Capabilities:

*   **Patient Data Ingestion**: Processes a patient’s discharge record and recent clinical history (demographics, diagnoses, labs, medications, prior visits).
*   **Data Validation**: Checks for missing or inconsistent data and flags them before analysis.
*   **Context Retrieval**: Retrieves supporting context such as hospital policy notes or readmission risk factors to ground the reasoning.
*   **Risk Estimation**: Estimates a patient’s 30-day readmission risk using a rule-based scoring model.
*   **Risk Categorization**: Assigns a risk category (low / medium / high) based on defined thresholds.
*   **Explanation Generation**: Provides a clear explanation of the main factors contributing to the risk score in clinician-friendly language.
*   **Structured Reporting**: Produces a structured report containing the risk score, risk level, key drivers, and suggested follow-up actions.
*   **Safety Checks**: Applies safety checks and confidence thresholds, routing high-risk or low-confidence cases for human review rather than autonomous action.
*   **Workflow Coordination**: Uses a supervisor agent to coordinate all steps via a state-driven workflow, ensuring controlled retries and full traceability of decisions.

## How it works:

1.  **Clarify Agent**: Handles missing critical information, asks clarifying questions, and flags cases requiring human review.
2.  **Validation Agent**: Validates raw patient data against a predefined schema (`PatientRecord`), captures errors, and routes invalid cases for review.
3.  **Rule-Based Risk Model Agent**: Calculates a 30-day readmission risk score and assigns a risk bucket (low/medium/high) and confidence level based on patient data.
4.  **Research Agent**: Retrieves general, non-patient-specific background context from sources like DuckDuckGo and Wikipedia to ground the LLM's explanations.
5.  **Explanation & Recommendation Agent**: Generates a clinician-friendly summary and non-prescriptive follow-up actions, ensuring compliance with `ClinicianReport` schema and safety guardrails.
6.  **Report Agent**: Formats the final output into a readable markdown string for display.
7.  **Supervisor Agent**: Orchestrates the entire workflow, manages state transitions, handles retries, and ensures safe progression through the system.

This system is designed for decision-support and does not provide medical advice or autonomous medical actions. All high-risk or low-confidence cases are explicitly routed for human clinician review.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

Writing README.md


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag